## 最简生成嵌入向量的代码
可用CPU运行

In [46]:
from transformers import BertTokenizer, BertModel

model_path = "bert-base-uncased"

# 要计算词向量的句子
text = "Hello, how are you?"

# 从HuggingFace Hub加载模型
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertModel.from_pretrained(model_path)

encoded_input = tokenizer(text, padding=True, truncation=True, return_tensors='pt')

# 计算标记嵌入
with torch.no_grad():
    model_output = model(**encoded_input, return_dict=True)

text_emb = model_output.pooler_output

In [59]:
model_output.last_hidden_state[:,:,0]

tensor([[-0.0824,  0.0310, -0.8935,  0.0393,  0.2573, -0.2812, -0.4429,  0.5570]])

In [57]:
model_output.pooler_output[:,0]

tensor([-0.9397])

## 使用Llama生成词向量
运行需要GPU

In [ ]:
from transformers import (
    LlamaModel,
    LlamaTokenizer,
    BitsAndBytesConfig,
)
import torch
from torch import clamp, sum

device = "auto"
model_path = "/mnt/chinese-llama-2-7b"             # Path to the combined weights

bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype="float16",
        bnb_4bit_use_double_quant=True,
    )

model = LlamaModel.from_pretrained(
    model_path, 
    device_map=device, 
    load_in_8bit= True,
    quantization_config=bnb_config
)
tokenizer = LlamaTokenizer.from_pretrained(model_path)

query = "Hello, how are you?"

encoded_input = tokenizer(query, padding=True, add_special_tokens = True, truncation = True, return_attention_mask = True, 
                          return_tensors='pt')

with torch.no_grad():
    model_output = model(**encoded_input, return_dict=True)

query_emb = model_output.last_hidden_state

### 池化 - 第一个标记

In [63]:
import torch.nn.functional as F

# 取第一个标记
first_token = model_output.last_hidden_state[:,0]

# 通过线性层
pooled_output = model.pooler.dense(first_token)

# 通过激活函数
pooled_output = F.tanh(pooled_output)


In [70]:
first_token[:,0]

tensor([-0.0824])

In [68]:
pooled_output[:,0]

tensor([-0.9397], grad_fn=<SelectBackward0>)

## 批量计算词向量

In [71]:
DATASET = 'squad'  # Huggingface Dataset to use
MODEL = 'bert-base-uncased'  # Transformer to use for embeddings
TOKENIZATION_BATCH_SIZE = 1000  # Batch size for tokenizing operaiton
INFERENCE_BATCH_SIZE = 64  # batch size for transformer
INSERT_RATIO = .001  # How many titles to embed and insert
DIMENSION = 768  # Embeddings size

from datasets import load_dataset, Dataset
from transformers import BertTokenizer, BertModel
 
data_dataset = load_dataset(DATASET, split='all')

# 生成一个固定的子集。为生成随机子集，请删除种子设置。有关详细信息，请参见 <https://huggingface.co/docs/datasets/v2.9.0/en/package_reference/main_classes#datasets.Dataset.train_test_split.seed>
data_dataset = data_dataset.train_test_split(test_size=INSERT_RATIO, seed=42)['test']

# 清理数据集中的数据结构
data_dataset = data_dataset.map(lambda val: {'answer': val['answers']['text'][0]}, remove_columns=['answers'])
 
tokenizer = BertTokenizer.from_pretrained(MODEL)
 
# 将问题分词以满足bert的输入格式。
def tokenize_question(batch):
    results = tokenizer(batch['question'], add_special_tokens = True, truncation = True, padding = True, # "max_length", \
                       return_tensors = "pt")
    batch['input_ids'] = results['input_ids']
    return batch
 
# 为每个条目生成标记
data_dataset = data_dataset.map(tokenize_question, batch_size=TOKENIZATION_BATCH_SIZE, batched=True)
# 设置输出格式为torch,以便将其推送到嵌入模型中
data_dataset.set_format('torch', columns=['input_ids'], output_all_columns=True)

model = BertModel.from_pretrained(MODEL)

def embed(batch):
    sentence_embs = model(input_ids=batch['input_ids']).pooler_output
    batch['question_embedding'] = sentence_embs
    return batch
    
data_dataset = data_dataset.map(embed, remove_columns=['input_ids'], batched = True, batch_size=INFERENCE_BATCH_SIZE)

def print_function(batch):
    print(batch['question'])
    print(batch['answer'])
    print(batch['question_embedding'])
 
data_dataset.map(print_function, batched=True, batch_size=4)

Found cached dataset squad (C:/Users/HP/.cache/huggingface/datasets/squad/plain_text/1.0.0/d6ec3ceb99ca480ce37cdd35555d6cb2511d223b9150cce08a837ef62ffea453)
Loading cached split indices for dataset at C:\Users\HP\.cache\huggingface\datasets\squad\plain_text\1.0.0\d6ec3ceb99ca480ce37cdd35555d6cb2511d223b9150cce08a837ef62ffea453\cache-3f8f639b545735e4.arrow and C:\Users\HP\.cache\huggingface\datasets\squad\plain_text\1.0.0\d6ec3ceb99ca480ce37cdd35555d6cb2511d223b9150cce08a837ef62ffea453\cache-8f60fd09160cab3e.arrow
Loading cached processed dataset at C:\Users\HP\.cache\huggingface\datasets\squad\plain_text\1.0.0\d6ec3ceb99ca480ce37cdd35555d6cb2511d223b9150cce08a837ef62ffea453\cache-092f53a95ba9ea09.arrow
Map:   0%|          | 0/99 [00:00<?, ? examples/s]We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
                                     

['Along with the United Democratic Party, what party currently rules the Marshall Islands?', "What was Gaddifi's original political viewpoint?", 'What is the name of the international airport in Guam?', 'Along with fishermen, what sort of Japanese people visited the Marshalls?']
['the AKA', 'Initially ideologically committed to Arab nationalism and Arab socialism', 'Antonio B. Won Pat', 'traders']
tensor([[-0.8324, -0.5737, -0.9900,  ..., -0.9467, -0.7632,  0.6736],
        [-0.6565, -0.4234, -0.9715,  ..., -0.8413, -0.6445,  0.5264],
        [-0.7713, -0.5848, -0.9829,  ..., -0.8983, -0.7207,  0.7519],
        [-0.6801, -0.5408, -0.9795,  ..., -0.9010, -0.7227,  0.5658]])
['For what would a bill be submitted to the Supreme Court of Scotland?', 'Which iPod model has its battery soldered into the unit?', 'Before he began studying economics, what concept did Hayek come up with that was later featured in his 1952 book?', 'What is the target demographic of the rhythmic oldies format?']
['r

Dataset({
    features: ['id', 'title', 'context', 'question', 'answer', 'input_ids', 'attention_mask', 'question_embedding'],
    num_rows: 99
})

## 池化操作

In [41]:
MODEL = 'bert-base-uncased'  # Transformer to use for embedding

from transformers import BertTokenizer, BertModel
from torch import clamp, sum

text = ["Around 9 Million empeople live in London", "London is known for its financial district"]

tokenizer = BertTokenizer.from_pretrained(MODEL)

encoded_input = tokenizer(text, padding=True, add_special_tokens = True, 
truncation = True, return_attention_mask = True, return_token_type_ids = True,
return_tensors='pt')
input_ids = encoded_input['input_ids']
token_type_ids = encoded_input['token_type_ids']
attention_mask = encoded_input['attention_mask']

# 计算标记嵌入
sentence_embs = model(input_ids, token_type_ids, attention_mask)[0]
input_mask_expanded = attention_mask.unsqueeze(-1).expand(sentence_embs.size()).float()
question_embedding = sum(sentence_embs * input_mask_expanded, 1) / clamp(input_mask_expanded.sum(1), min=1e-9)


In [39]:
question_embedding.shape

torch.Size([1, 768])

In [42]:
attention_mask

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]])

In [45]:
input_mask_expanded

tensor([[[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]])

In [46]:
question_embedding

tensor([[ 0.3993,  0.4458,  0.2752,  ..., -0.7604,  0.0992,  0.2496],
        [-0.0432,  0.2758,  0.1875,  ..., -0.3111,  0.0233, -0.0341]],
       grad_fn=<DivBackward0>)

## 文档搜索案例

In [34]:
from transformers import BertTokenizer, BertModel
import torch

MODEL = 'bert-base-uncased'  # Transformer to use for embeddings

#直接读取池化后的词向量
def encode(texts):
    # Tokenize sentences
    encoded_input = tokenizer(texts, padding=True, add_special_tokens = True, truncation = True, return_attention_mask = True, 
                          return_token_type_ids = True, return_tensors='pt')
    input_ids = encoded_input['input_ids']
    
    # 计算标记嵌入
    with torch.no_grad():   
        sentence_embs = model(input_ids).pooler_output

    return sentence_embs


#要嵌入的语句
#query = "How many people live in London?"
#docs = ["Around 9 Million empeople live in London", "London is known for its financial district"]

query = "我想喝一杯浓郁的咖啡。"
docs = ["我今天早上喝了一杯香浓的咖啡。","我喜欢喝美式咖啡。","今天的天气真好！"] 

model = BertModel.from_pretrained(MODEL)
tokenizer = BertTokenizer.from_pretrained(MODEL)

#计算query和docs的词向量
query_emb = encode(query)
doc_emb = encode(docs)

#计算L2距离
scores = torch.norm(query_emb - doc_emb, p=2, dim=1) 

#组合分数和文档语句
doc_score_pairs = list(zip(docs, scores))

#按分数倒排序
doc_score_pairs = sorted(doc_score_pairs, key=lambda x: x[1], reverse=True)

#输出分数
for doc, score in doc_score_pairs:
    print(score, doc)

tensor(5.7056) 今天的天气真好！
tensor(4.5422) 我喜欢喝美式咖啡。
tensor(1.1916) 我今天早上喝了一杯香浓的咖啡。
